In [ ]:
import pickle

import polars as pl

FOLIO_RAW_BIBS_PATH = "data/folio-raw-bibs.parquet"
FOLIO_RAW_ITEMS_PATH = "data/folio-raw-items.parquet"
FOLIO_SOURCE_WORKS_PATH = "data/folio-source-works.parquet"

def save_parquet_snapshot(data: dict, path: str):
    df = pl.DataFrame({
        "id": list(data.keys()),
        "body": [pickle.dumps(v) for v in data.values()],
    })
    df.write_parquet(path)

folio_raw_bibs = pl.read_parquet(
    FOLIO_RAW_BIBS_PATH, columns=["id", "content", "last_modified", "deleted"]
)
# Item/holdings enrichment, joined onto each bib by id as FolioStoreSource does.
folio_raw_items = dict(
    pl.read_parquet(FOLIO_RAW_ITEMS_PATH, columns=["id", "content"]).iter_rows()
)

print(len(folio_raw_bibs), "bibs,", len(folio_raw_items), "enriched instances")

In [ ]:
from collections import Counter

from adapters.transformers.builders.folio_work_builder import FolioWorkBuilder
from adapters.transformers.marc.identifier import has_id
from ingestor.models.shared.deleted_reason import DeletedFromSource
from utils.marc import parse_single_marc_record

transformed_folio_works = {}
missing_content = []
unparseable = []
missing_id_field = []
failed = []
# FolioWorkBuilder turns tombstones and suppressed instances into DeletedSourceWork
# rather than raising, so they are counted rather than bucketed as errors.
deleted = 0

for i, row in enumerate(folio_raw_bibs.iter_rows(named=True)):
    work_id = row["id"]

    if not row["content"]:
        missing_content.append(work_id)
        continue

    try:
        record = parse_single_marc_record(row["content"])
    except Exception:
        unparseable.append(work_id)
        continue

    # A record with no 001 cannot be processed for any source, so skip it.
    if not has_id(record):
        missing_id_field.append(work_id)
        continue

    try:
        builder = FolioWorkBuilder(
            record=record,
            last_modified=row["last_modified"],
            enrichment_content=folio_raw_items.get(work_id),
        )
        if row["deleted"]:
            folio_work = builder.transform_deleted_work(deleted_reason=DeletedFromSource())
        else:
            folio_work = builder.transform_work()
        transformed_folio_works[work_id] = folio_work
        deleted += folio_work.type == "Deleted"
    except Exception as e:
        failed.append((work_id, str(e)))

    if i % 20000 == 0:
        print(f"Progress: {i}")

print("Successfully transformed works:", len(transformed_folio_works))
print("  of which deleted or suppressed:", deleted)
print("Folio records with no content:", len(missing_content))
print("Folio records with unparseable MARC:", len(unparseable))
print("Folio records with missing ID field:", len(missing_id_field))
print("Folio records that failed to transform:", len(failed))
for message, count in Counter(error for _, error in failed).most_common(10):
    print(f"  {count:>7}  {message}")

In [ ]:
save_parquet_snapshot(transformed_folio_works, FOLIO_SOURCE_WORKS_PATH)